In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, precision_score, recall_score
import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import os
import json
import pickle

In [ ]:
# Load Yijun's 5-fold indices
kfold_pt = '.'
with open(kfold_pt, 'r') as file:
    kfold_indices = json.load(file)
print(type(kfold_indices))

KFOLD INDICES STRUCTURE:
Dict
- first level are keys labeled fold_0 to fold_1
    - in each fold_i value is another dictionary where the keys are 'train' and 'test'
        - the value is a list of indices

In [ ]:
# Load in the geojson with data
pt_data_pt = '..'
point_data_gdf = gpd.read_file(pt_data_pt)
print(point_data_gdf.columns)

In [ ]:
# Define random forest run function
def run_rf(all_ids, kfold_indices, X, y, hyperparameters, save_preds=False, save_rf=False):
    indices = np.array(all_ids)
    
    fold_accuracies = []
    fold_precisions = []
    fold_recalls = []
    fold_precision_0 = []
    fold_precision_1 = []
    fold_recall_0 = []
    fold_recall_1 = []
    
    for fold, train_test_dict in kfold_indices.items():
        print('Training Fold: ', fold)

        train_indices = train_test_dict['train']
        test_indices = train_test_dict['test']

        train_rows = np.where(np.isin(indices, train_indices))[0]
        test_rows = np.where(np.isin(indices, test_indices))[0]

        X_train = X[train_rows]
        X_test = X[test_rows]
        y_train = y[train_rows]
        y_test = y[test_rows] 

        print(f"X_train shape: {X_train.shape}, X_test shape: {X_test.shape}")
        print(f"y_train shape: {y_train.shape}, y_test shape: {y_test.shape}")

        est = hyperparameters['est']
        verbose = hyperparameters['verbose']
        n_jobs = hyperparameters['n_jobs']
        rf = RandomForestClassifier(n_estimators=est, verbose=verbose, n_jobs=n_jobs)
        rf.fit(X_train, y_train.ravel())
        y_pred = rf.predict(X_test)

        accuracy = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred, average='binary')
        recall = recall_score(y_test, y_pred, average='binary')
        class_precisions = precision_score(y_test, y_pred, average=None)
        class_recalls = recall_score(y_test, y_pred, average=None)
        
        fold_accuracies.append(accuracy)
        fold_precisions.append(precision)
        fold_recalls.append(recall)
        fold_precision_0.append(class_precisions[0])
        fold_precision_1.append(class_precisions[1])
        fold_recall_0.append(class_recalls[0])
        fold_recall_1.append(class_recalls[1])
        
        fold_num = int(fold.split('_')[-1])
        print(f"Fold {fold_num} Accuracy: {accuracy:.4f}")
        print(f"Fold {fold_num} Precision: {precision:.4f}")
        print(f"Fold {fold_num} Recall: {recall:.4f}")
        
        if save_preds:
            test_ids = indices[np.isin(indices, test_indices)]
            print(len(test_ids))
            print(len(test_indices))
            preds = {
                "id": test_ids,  
                "gt": y_test.flatten(), 
                "pred": y_pred  
            }
            df_predictions = pd.DataFrame(preds)
            preds_outpath = f"test_predictions_{fold}.csv"
            df_predictions.to_csv(preds_outpath, index=False)

            print(f"Test predictions saved to {preds_outpath}")
            
        if save_rf:
            model_file = f"rf_weights_{fold}.pkl"
            with open(model_file, "wb") as f:
                pickle.dump(rf, f)
            print(f"Model saved to {model_file}")
        
    return (
        fold_accuracies, fold_precisions, fold_recalls,
        fold_precision_0, fold_precision_1,
        fold_recall_0, fold_recall_1
    )

## Experiment Random Forest 1 (RF-1)
### Settings:
- Spatial Scale: Pixel Level (i.e. no buffer)
- Inputs: 
    - Channels: R,G,B
    - Covariates: Lat, Lon
    - Order: B, G, R, Lon, Lat
- Output: 
    - Presence/Absence of Permafrost, aksdb_pf1m_bin (image channel), binary classification
- Preprocessing: None


In [ ]:
print('Before removing NaN: ', point_data_gdf.shape)
point_data_gdf = point_data_gdf.dropna(subset=['aksdb_pf1m_bin_gt']).reset_index(drop=True)
print('After removing NaN: ', point_data_gdf.shape)

In [ ]:
# Loading in data 
channel_names = ['band_25', 'band_26', 'band_27']
covariates = ['lon', 'lat']
x_vars = channel_names + covariates
y_var = ['aksdb_pf1m_bin_gt']

X = point_data_gdf[x_vars].to_numpy()
y = point_data_gdf[y_var].to_numpy()
print('X shape: ', X.shape)
print('Y shape: ', y.shape)

#load the order of the indices
indices = point_data_gdf['id'].to_list()

In [ ]:
#Initialize RF model hyperparameters
hyperparameters = {
    'est': 100,
    'n_jobs': 4,
    'verbose': 1
}

In [ ]:
# Run specific experiment settings
# indices, X, y, and hyperparameters should be specific to the variable of interest
# kfold_indices should remain the same among all experiments
(
    accuracy, precision, recall,
    precision_0, precision_1,
    recall_0, recall_1
) = run_rf(indices, kfold_indices, X, y, hyperparameters, save_preds = False, save_rf=False)

results = {
    "Fold": list(range(0, len(accuracy))),
    "Accuracy": accuracy,
    "Precision": precision,
    "Recall": recall,
    "Precision_0": precision_0,
    "Recall_0": recall_0,
    "Precision_1": precision_1,
    "Recall_1": recall_1,
    
}
results_df = pd.DataFrame(results)
result_outpath = "results/SFold_PermafrostPrediction/RF-1_results.csv"
results_df.to_csv(result_outpath, index = False)
print(f"Metrics saved to {result_outpath}")

## Experiment Random Forest 2 (RF-2)
### Settings:
- Spatial Scale: Pixel Level (i.e. no buffer)
- Inputs: 
    - Channels: 25, 26, 27, 28, 29, 30, 31, 33, 34
    - Covariates: Lat, Lon
    - Intuition: Basically all the midsummer channels
    - Order: B, G, R, rededge1, rededge2, rededge3, nir, swir1, swir2, lon, lat
- Output: 
    - Presence/Absence of Permafrost, aksdb_pf1m_bin (image channel), binary classification
- Preprocessing: None

In [ ]:
print('Before removing NaN: ', point_data_gdf.shape)
point_data_gdf = point_data_gdf.dropna(subset=['aksdb_pf1m_bin_gt']).reset_index(drop=True)
print('After removing NaN: ', point_data_gdf.shape)

In [ ]:
# Loading in data 
channel_names = ['band_25', 'band_26', 'band_27', 'band_28', 'band_29', 'band_30', 'band_31', 'band_33', 'band_34']
covariates = ['lon', 'lat']
x_vars = channel_names + covariates
y_var = ['aksdb_pf1m_bin_gt']

X = point_data_gdf[x_vars].to_numpy()
y = point_data_gdf[y_var].to_numpy()
print('X shape: ', X.shape)
print('Y shape: ', y.shape)

#load the order of the indices
indices = point_data_gdf['id'].to_list()

In [ ]:
#Initialize RF model hyperparameters
hyperparameters = {
    'est': 100,
    'n_jobs': 4,
    'verbose': 1
}

In [ ]:
# Run specific experiment settings
# indices, X, y, and hyperparameters should be specific to the variable of interest
# kfold_indices should remain the same among all experiments
(
    accuracy, precision, recall,
    precision_0, precision_1,
    recall_0, recall_1
) = run_rf(indices, kfold_indices, X, y, hyperparameters, save_preds = True, save_rf=True)

results = {
    "Fold": list(range(0, len(accuracy))),
    "Accuracy": accuracy,
    "Precision": precision,
    "Recall": recall,
    "Precision_0": precision_0,
    "Recall_0": recall_0,
    "Precision_1": precision_1,
    "Recall_1": recall_1,
    
}
results_df = pd.DataFrame(results)
result_outpath = "results/SFold/RF-2_results.csv"
results_df.to_csv(result_outpath, index = False)
print(f"Metrics saved to {result_outpath}")

## Experiment Random Forest 3 (RF-3)
### Settings:
- Spatial Scale: Pixel Level (i.e. no buffer)
- Inputs: 
    - Channels: 25, 26, 27, (bands['s2_1_nir'] - bands['s2_1_red']) / (bands['s2_1_nir'] + bands['s2_1_red'])
    - Covariates: Lat, Lon
    - Order: B, G, R, NDVI
    - Intuition: Best performing channels + one natural derivative
- Output: 
    - Presence/Absence of Permafrost, aksdb_pf1m_bin (image channel), binary classification
- Preprocessing: None

In [ ]:
print('Before removing NaN: ', point_data_gdf.shape)
point_data_gdf = point_data_gdf.dropna(subset=['aksdb_pf1m_bin_gt']).reset_index(drop=True)
print('After removing NaN: ', point_data_gdf.shape)

In [ ]:
#Calculating the derivative channels
point_data_gdf['ndvi'] = (point_data_gdf['band_31'] - point_data_gdf['band_27']) / (point_data_gdf['band_31'] + point_data_gdf['band_27'])

In [ ]:
# Loading in data 
channel_names = ['band_25', 'band_26', 'band_27', 'ndvi']
covariates = ['lon', 'lat']
x_vars = channel_names + covariates
y_var = ['aksdb_pf1m_bin_gt']

X = point_data_gdf[x_vars].to_numpy()
y = point_data_gdf[y_var].to_numpy()
print('X shape: ', X.shape)
print('Y shape: ', y.shape)

#load the order of the indices
indices = point_data_gdf['id'].to_list()

In [ ]:
#Initialize RF model hyperparameters
hyperparameters = {
    'est': 100,
    'n_jobs': 4,
    'verbose': 1
}

In [ ]:
# Run specific experiment settings
# indices, X, y, and hyperparameters should be specific to the variable of interest
# kfold_indices should remain the same among all experiments
(
    accuracy, precision, recall,
    precision_0, precision_1,
    recall_0, recall_1
) = run_rf(indices, kfold_indices, X, y, hyperparameters, save_preds = False, save_rf=False)

results = {
    "Fold": list(range(0, len(accuracy))),
    "Accuracy": accuracy,
    "Precision": precision,
    "Recall": recall,
    "Precision_0": precision_0,
    "Recall_0": recall_0,
    "Precision_1": precision_1,
    "Recall_1": recall_1,
    
}
results_df = pd.DataFrame(results)
result_outpath = "results/RF-3_results.csv"
results_df.to_csv(result_outpath, index = False)
print(f"Metrics saved to {result_outpath}")

## Experiment Random Forest 4 (RF-4)
### Settings:
- Spatial Scale: Pixel Level (i.e. no buffer)
- Inputs: 
    - Channels: 25, 26, 27,
                (bands['s2_1_nir'] - bands['s2_1_swir2']) / (bands['s2_1_nir'] + bands['s2_1_swir2']),
                (bands['s2_1_green'] - bands['s2_1_red']) / (bands['s2_1_green'] + bands['s2_1_red']),
                (bands['s2_1_nir'] - bands['s2_1_swir1']) / (bands['s2_1_nir'] + bands['s2_1_swir1']),
                (bands['s2_1_green'] - bands['s2_1_swir1']) / (bands['s2_1_green'] + bands['s2_1_swir1']),
                (bands['s2_1_nir'] - bands['s2_1_red']) / (bands['s2_1_nir'] + bands['s2_1_red']),
                (bands['s2_1_green'] - bands['s2_1_nir']) / (bands['s2_1_green'] + bands['s2_1_nir'])
    - Covariates: Lat, Lon
    - Order: B, G, R, NBR, NGRDI, NDMI, NDSI, NDVI, NDWI, Lon, Lat
    - Intuition: Best performing channels + all derivatives
- Output: 
    - Presence/Absence of Permafrost, aksdb_pf1m_bin (image channel), binary classification
- Preprocessing: None

In [ ]:
print('Before removing NaN: ', point_data_gdf.shape)
point_data_gdf = point_data_gdf.dropna(subset=['aksdb_pf1m_bin_gt']).reset_index(drop=True)
print('After removing NaN: ', point_data_gdf.shape)

In [ ]:
#Calculating the derivative channels
point_data_gdf['nbr'] = (point_data_gdf['band_31'] - point_data_gdf['band_34']) / (point_data_gdf['band_31'] + point_data_gdf['band_34'])
point_data_gdf['ngrdi'] = (point_data_gdf['band_26'] - point_data_gdf['band_27']) / (point_data_gdf['band_26'] + point_data_gdf['band_27'])
point_data_gdf['ndmi'] = (point_data_gdf['band_31'] - point_data_gdf['band_33']) / (point_data_gdf['band_31'] + point_data_gdf['band_33'])
point_data_gdf['ndsi'] = (point_data_gdf['band_26'] - point_data_gdf['band_33']) / (point_data_gdf['band_26'] + point_data_gdf['band_33'])
point_data_gdf['ndvi'] = (point_data_gdf['band_31'] - point_data_gdf['band_27']) / (point_data_gdf['band_31'] + point_data_gdf['band_27'])
point_data_gdf['ndwi'] = (point_data_gdf['band_26'] - point_data_gdf['band_31']) / (point_data_gdf['band_26'] + point_data_gdf['band_31'])

In [ ]:
# Loading in data 
channel_names = ['band_25', 'band_26', 'band_27', 'nbr', 'ngrdi', 'ndmi', 'ndsi', 'ndvi', 'ndwi']
covariates = ['lon', 'lat']
x_vars = channel_names + covariates
y_var = ['aksdb_pf1m_bin_gt']

X = point_data_gdf[x_vars].to_numpy()
y = point_data_gdf[y_var].to_numpy()
print('X shape: ', X.shape)
print('Y shape: ', y.shape)

#load the order of the indices
indices = point_data_gdf['id'].to_list()

In [ ]:
# Run specific experiment settings
# indices, X, y, and hyperparameters should be specific to the variable of interest
# kfold_indices should remain the same among all experiments
(
    accuracy, precision, recall,
    precision_0, precision_1,
    recall_0, recall_1
) = run_rf(indices, kfold_indices, X, y, hyperparameters, save_preds = False, save_rf=False)

results = {
    "Fold": list(range(0, len(accuracy))),
    "Accuracy": accuracy,
    "Precision": precision,
    "Recall": recall,
    "Precision_0": precision_0,
    "Recall_0": recall_0,
    "Precision_1": precision_1,
    "Recall_1": recall_1,
    
}
results_df = pd.DataFrame(results)
result_outpath = "results/SFold_PermafrostPrediction/RF-4_results.csv"
results_df.to_csv(result_outpath, index = False)
print(f"Metrics saved to {result_outpath}")

## Experiment Random Forest 5 (RF-5)
### Settings:
- Spatial Scale: Pixel Level (i.e. no buffer)
- Inputs: 
    - Channels: None
    - Covariates: Lat, Lon
    - Order: N/A
    - Intuition: Just Lat/Lon
- Output: 
    - Presence/Absence of Permafrost, aksdb_pf1m_bin (image channel), binary classification
- Preprocessing: None

In [ ]:
print('Before removing NaN: ', point_data_gdf.shape)
point_data_gdf = point_data_gdf.dropna(subset=['aksdb_pf1m_bin_gt']).reset_index(drop=True)
print('After removing NaN: ', point_data_gdf.shape)

In [ ]:
# Loading in data 
channel_names = []
covariates = ['lon', 'lat']
x_vars = channel_names + covariates
y_var = ['aksdb_pf1m_bin_gt']

X = point_data_gdf[x_vars].to_numpy()
y = point_data_gdf[y_var].to_numpy()
print('X shape: ', X.shape)
print('Y shape: ', y.shape)

#load the order of the indices
indices = point_data_gdf['id'].to_list()

In [ ]:
# Run specific experiment settings
# indices, X, y, and hyperparameters should be specific to the variable of interest
# kfold_indices should remain the same among all experiments
(
    accuracy, precision, recall,
    precision_0, precision_1,
    recall_0, recall_1
) = run_rf(indices, kfold_indices, X, y, hyperparameters, save_preds = False, save_rf=False)

results = {
    "Fold": list(range(0, len(accuracy))),
    "Accuracy": accuracy,
    "Precision": precision,
    "Recall": recall,
    "Precision_0": precision_0,
    "Recall_0": recall_0,
    "Precision_1": precision_1,
    "Recall_1": recall_1,
    
}
results_df = pd.DataFrame(results)
result_outpath = "results/SFold_PermafrostPrediction/RF-5_results.csv"
results_df.to_csv(result_outpath, index = False)
print(f"Metrics saved to {result_outpath}")

## Experiment Random Forest 6 (RF-6)
### Settings:
- Spatial Scale: Buffer level - 100 m
- Inputs: 
    - Channels: R,G,B
    - Covariates: Lat, Lon (center)
    - Order: B, G, R, Lat, Lon
- Output: 
    - Presence/Absence of Permafrost, aksdb_pf1m_bin (image channel), binary classification
- Preprocessing: None


In [ ]:
print('Before removing NaN: ', point_data_gdf.shape)
point_data_gdf = point_data_gdf.dropna(subset=['aksdb_pf1m_bin_gt']).reset_index(drop=True)
print('After removing NaN: ', point_data_gdf.shape)

In [ ]:
# Project gdf to EPSG 3338 since we are going to do buffer operations
point_data_gdf = point_data_gdf.to_crs(epsg=3338)

In [ ]:
# Loading in data
# For every row in the dataframe, get all the rows that are within a 100 meter buffer
# For all the rows in a 100 meter buffer get the average r, average g, and average b
point_data_gdf['100m_buffer'] = point_data_gdf.geometry.buffer(100)
buffers_gdf = point_data_gdf.set_geometry('100m_buffer')

sjoin_result = gpd.sjoin(point_data_gdf, buffers_gdf[['100m_buffer']], predicate="within")

# Group by buffer geometry and calculate the mean for each band
band_means = (
    sjoin_result.groupby(sjoin_result.index_right)[['band_25', 'band_26', 'band_27']]
    .mean()
    .reset_index()
)

# Rename columns for clarity
band_means = band_means.rename(
    columns={
        'band_25': 'ave_band_25',
        'band_26': 'ave_band_26',
        'band_27': 'ave_band_27'
    }
)

# Merge the calculated averages back into the buffers GeoDataFrame
buffers_points_gdf = buffers_gdf.merge(band_means, left_index=True, right_on='index_right', how='left')
print(buffers_points_gdf.columns)
# averages = (
#     buffered_gdf.groupby('index_left')
#     .agg(avg_r=('r', 'mean'), avg_g=('g', 'mean'), avg_b=('b', 'mean'))
# )
# point_data_gdf = point_data_gdf.join(averages)
# print(point_data_gdf.head(2))

In [ ]:
# Loading in data 
channel_names = ['ave_band_25', 'ave_band_26', 'ave_band_27']
covariates = ['lon', 'lat']
x_vars = channel_names + covariates
y_var = ['aksdb_pf1m_bin_gt']

X = buffers_points_gdf[x_vars].to_numpy()
y = buffers_points_gdf[y_var].to_numpy()
print('X shape: ', X.shape)
print('Y shape: ', y.shape)

#load the order of the indices
indices = buffers_points_gdf['id'].to_list()

In [ ]:
#Initialize RF model hyperparameters
hyperparameters = {
    'est': 100,
    'n_jobs': 4,
    'verbose': 1
}

In [ ]:
# Run specific experiment settings
# indices, X, y, and hyperparameters should be specific to the variable of interest
# kfold_indices should remain the same among all experiments
(
    accuracy, precision, recall,
    precision_0, precision_1,
    recall_0, recall_1
) = run_rf(indices, kfold_indices, X, y, hyperparameters, save_preds = False, save_rf=False)

results = {
    "Fold": list(range(0, len(accuracy))),
    "Accuracy": accuracy,
    "Precision": precision,
    "Recall": recall,
    "Precision_0": precision_0,
    "Recall_0": recall_0,
    "Precision_1": precision_1,
    "Recall_1": recall_1,
    
}
results_df = pd.DataFrame(results)
result_outpath = "results/RF-6_results.csv"
results_df.to_csv(result_outpath, index = False)
print(f"Metrics saved to {result_outpath}")

## Experiment Random Forest 7 (RF-7)
### Settings:
- Spatial Scale: Pixel Level (i.e. no buffer)
- Inputs: 
    - Channels: R,G,B, aspect_4, elevation_full_10m, maxc_4, sl_4, spi, swi_10, tpi_4, Lat, Lon
    - Covariates: Lat, Lon
    - Order: B, G, R, aspect_4, elevation_full_10m, maxc_4, sl_4, spi, swi_10, tpi_4, Lon, Lat
- Output: 
    - Permafrost presence prediction
- Preprocessing: removal of NoData portions of sentinel data


In [ ]:
print('Before removing NaN: ', point_data_gdf.shape)
point_data_gdf = point_data_gdf.dropna(subset=['aksdb_pf1m_bin_gt']).reset_index(drop=True)
print('After removing NaN: ', point_data_gdf.shape)

In [ ]:
# Loading in covars
covar_pt = '../data/point_pixel_ifsar_derivs_v1.csv'
covar_gdf = gpd.read_file(covar_pt)
print(covar_gdf.columns)
covar_gdf = covar_gdf[['id', 'aspct_4_band_1', 'elevation_full_10m_3338_band_1', 'maxc_4_band_1', 'sl_4_band_1', 'spi_band_1', 'swi_10_band_1', 'tpi_4_band_1']]

In [ ]:
point_data_gdf['id'] = point_data_gdf['id'].astype('int64')
covar_gdf['id'] = covar_gdf['id'].astype('int64')
merged_df = pd.merge(point_data_gdf, covar_gdf, on='id', how='inner')

In [ ]:
# Loading in data 
channel_names = ['band_25', 'band_26', 'band_27', 'aspct_4_band_1', 'elevation_full_10m_3338_band_1', 'maxc_4_band_1', 'sl_4_band_1', 'spi_band_1', 'swi_10_band_1', 'tpi_4_band_1']
covariates = ['lon', 'lat']
x_vars = channel_names + covariates
y_var = ['aksdb_pf1m_bin_gt']

X = merged_df[x_vars].to_numpy()
y = merged_df[y_var].to_numpy()
print('X shape: ', X.shape)
print('Y shape: ', y.shape)

#load the order of the indices
indices = point_data_gdf['id'].to_list()

In [ ]:
#Initialize RF model hyperparameters
hyperparameters = {
    'est': 100,
    'n_jobs': 4,
    'verbose': 1
}

In [ ]:
# Run specific experiment settings
# indices, X, y, and hyperparameters should be specific to the variable of interest
# kfold_indices should remain the same among all experiments
(
    accuracy, precision, recall,
    precision_0, precision_1,
    recall_0, recall_1
) = run_rf(indices, kfold_indices, X, y, hyperparameters, save_preds = False, save_rf=False)

results = {
    "Fold": list(range(0, len(accuracy))),
    "Accuracy": accuracy,
    "Precision": precision,
    "Recall": recall,
    "Precision_0": precision_0,
    "Recall_0": recall_0,
    "Precision_1": precision_1,
    "Recall_1": recall_1,
    
}
results_df = pd.DataFrame(results)
result_outpath = "results/SFold_PermafrostPrediction/RF-7_results.csv"
results_df.to_csv(result_outpath, index = False)
print(f"Metrics saved to {result_outpath}")

## Experiment Random Forest 11 (RF-11)
### Settings:
- Spatial Scale: Pixel Level (i.e. no buffer)
- Inputs: 
    - Channels: 25, 26, 27, 28, 29, 30, 31, 33, 34, aspect_4, elevation_full_10m, maxc_4, sl_4, spi, swi_10, tpi_4, ppt_annual, tmean_swi, tmin_january
    - Covariates: Lat, Lon
    - Order: basically satellite channels, env covars like DEM, climate covars
        - B, G, R, rededge1, rededge2, rededge3, nir, swir1, aspect_4, elevation_full_10m, maxc_4, sl_4, spi, swi_10, tpi_4, ppt_annual, tmean_swi, tmin_january, Lon, Lat
- Output: 
    - Presence/Absence of Permafrost, aksdb_pf1m_bin (image channel), binary classification
- Preprocessing: removal of NoData rows

In [ ]:
print('Before removing NaN: ', point_data_gdf.shape)
point_data_gdf = point_data_gdf.dropna(subset=['aksdb_pf1m_bin_gt']).reset_index(drop=True)
print('After removing NaN: ', point_data_gdf.shape)

In [ ]:
# Loading in covars
topo_covar_pt = '../data/point_pixel_ifsar_derivs_v1.csv'
topo_covar_gdf = gpd.read_file(topo_covar_pt)
topo_covar_gdf = topo_covar_gdf[['id', 'aspct_4_band_1', 'elevation_full_10m_3338_band_1', 'maxc_4_band_1', 'sl_4_band_1', 'spi_band_1', 'swi_10_band_1', 'tpi_4_band_1']]
topo_covar_gdf['id'] = topo_covar_gdf['id'].astype('int64')

climate_covar_pt = '../data/point_pixel_climate_v1.csv'
climate_covar_gdf = pd.read_csv(climate_covar_pt)
climate_covar_gdf = climate_covar_gdf[['id', 'ppt_annual_band_1', 'tmean_swi_band_1', 'tmin_january_band_1']]

In [ ]:
merged_df = point_data_gdf.merge(topo_covar_gdf, on='id', how='inner')
merged_df = merged_df.merge(climate_covar_gdf, on='id', how='inner')

In [ ]:
# Loading in data
channel_names = ['band_25', 'band_26', 'band_27', 'band_28', 'band_29', 'band_30', 'band_31', 'band_33', 'band_34', 
                    'aspct_4_band_1', 'elevation_full_10m_3338_band_1', 'maxc_4_band_1', 'sl_4_band_1', 'spi_band_1', 
                     'swi_10_band_1', 'tpi_4_band_1', 'ppt_annual_band_1', 'tmean_swi_band_1', 'tmin_january_band_1']
covariates = ['lon', 'lat']
x_vars = channel_names + covariates
y_var = ['aksdb_pf1m_bin_gt']

X = merged_df[x_vars].to_numpy()
y = merged_df[y_var].to_numpy()
print('X shape: ', X.shape)
print('Y shape: ', y.shape)

indices = merged_df['id'].to_list()

In [ ]:
#Initialize RF model hyperparameters
hyperparameters = {
    'est': 100,
    'n_jobs': 4,
    'verbose': 1
}

In [ ]:
# Run specific experiment settings
# indices, X, y, and hyperparameters should be specific to the variable of interest
# kfold_indices should remain the same among all experiments
(
    accuracy, precision, recall,
    precision_0, precision_1,
    recall_0, recall_1
) = run_rf(indices, kfold_indices, X, y, hyperparameters, save_preds = False, save_rf=True)

results = {
    "Fold": list(range(0, len(accuracy))),
    "Accuracy": accuracy,
    "Precision": precision,
    "Recall": recall,
    "Precision_0": precision_0,
    "Recall_0": recall_0,
    "Precision_1": precision_1,
    "Recall_1": recall_1,
    
}
results_df = pd.DataFrame(results)
result_outpath = "results/SFold_PermafrostPrediction/RF-11_results.csv"
results_df.to_csv(result_outpath, index = False)
print(f"Metrics saved to {result_outpath}")